# VulnScanner AI — ML Model Training Notebook

This notebook walks through the complete ML pipeline:
1. Dataset generation and exploration
2. Feature extraction and analysis
3. Model training with cross-validation
4. Performance evaluation and visualisation
5. SHAP explainability analysis
6. Model export

**Run from the project root:** `jupyter notebook notebooks/model_training.ipynb`

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0d1117'
matplotlib.rcParams['axes.facecolor'] = '#161b22'
matplotlib.rcParams['axes.edgecolor'] = '#30363d'
matplotlib.rcParams['text.color'] = '#e6edf3'
matplotlib.rcParams['axes.labelcolor'] = '#8b949e'
matplotlib.rcParams['xtick.color'] = '#8b949e'
matplotlib.rcParams['ytick.color'] = '#8b949e'
matplotlib.rcParams['grid.color'] = '#30363d'

print('Environment ready ✓')

## 1. Dataset Generation

In [ ]:
from vulnscanner.ml.trainer import generate_synthetic_dataset

df = generate_synthetic_dataset()
print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['label'].value_counts().rename({0: 'Safe', 1: 'Vulnerable'}).to_string())
print(f'\nLanguage distribution:')
print(df['language'].value_counts().to_string())
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df['label'].value_counts()
axes[0].bar(['Safe', 'Vulnerable'], counts.values,
            color=['#2ed573', '#ff4757'], edgecolor='#30363d', linewidth=0.5)
axes[0].set_title('Class Distribution', color='#58a6ff', fontsize=13)
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 0.3, str(v), ha='center', color='#e6edf3')

# Language distribution
lang_counts = df['language'].value_counts()
colors = ['#58a6ff', '#bc8cff', '#ff6b35', '#2ed573', '#ffa502']
axes[1].bar(lang_counts.index, lang_counts.values,
            color=colors[:len(lang_counts)], edgecolor='#30363d', linewidth=0.5)
axes[1].set_title('Language Distribution', color='#58a6ff', fontsize=13)
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('../datasets/dataset_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Chart saved to datasets/dataset_distribution.png')

## 2. Feature Extraction & Analysis

In [ ]:
from vulnscanner.ml.features import FeatureExtractor, FeatureVector

extractor = FeatureExtractor()
feature_names = FeatureVector.feature_names()

# Extract features for all samples
pairs = list(zip(df['code_snippet'].tolist(), df['language'].tolist()))
X = extractor.extract_batch(pairs)
y = df['label'].values

print(f'Feature matrix shape: {X.shape}')
print(f'Feature names ({len(feature_names)}): {feature_names}')

# Feature statistics by class
feat_df = pd.DataFrame(X, columns=feature_names)
feat_df['label'] = y
print('\nMean feature values by class:')
feat_df.groupby('label').mean().T.rename(columns={0: 'Safe', 1: 'Vulnerable'}).round(3)

In [ ]:
# Feature importance heatmap — mean difference between classes
safe_means = feat_df[feat_df['label'] == 0][feature_names].mean()
vuln_means = feat_df[feat_df['label'] == 1][feature_names].mean()
diff = (vuln_means - safe_means).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#ff4757' if v > 0 else '#2ed573' for v in diff.values]
bars = ax.barh(diff.index, diff.values, color=colors, edgecolor='#30363d', linewidth=0.5)
ax.axvline(0, color='#484f58', linewidth=1)
ax.set_title('Feature Discriminative Power\n(Vulnerable mean − Safe mean)',
             color='#58a6ff', fontsize=13)
ax.set_xlabel('Mean Difference')
plt.tight_layout()
plt.savefig('../datasets/feature_importance.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

## 3. Model Training with Cross-Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, learning_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score
)

try:
    from xgboost import XGBClassifier
    model = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42, n_jobs=-1
    )
    model_name = 'XGBoost'
except ImportError:
    from sklearn.ensemble import GradientBoostingClassifier
    model = GradientBoostingClassifier(n_estimators=200, max_depth=5, random_state=42)
    model_name = 'GradientBoosting'

print(f'Training model: {model_name}')

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
cv_f1  = cross_val_score(model, X, y, cv=cv, scoring='f1')
cv_acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy')

print(f'\n5-Fold Cross-Validation Results:')
print(f'  ROC-AUC:  {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')
print(f'  F1 Score: {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'  Accuracy: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}')

In [ ]:
# Train final model on all data
model.fit(X, y)
y_pred = model.predict(X)
y_proba = model.predict_proba(X)[:, 1]

print('Classification Report (training set):')
print(classification_report(y, y_pred, target_names=['Safe', 'Vulnerable']))
print(f'Train ROC-AUC: {roc_auc_score(y, y_proba):.4f}')

## 4. Performance Visualisation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── ROC Curve ──
fpr, tpr, _ = roc_curve(y, y_proba)
auc = roc_auc_score(y, y_proba)
axes[0].plot(fpr, tpr, color='#58a6ff', lw=2, label=f'ROC (AUC = {auc:.3f})')
axes[0].plot([0,1],[0,1], color='#484f58', lw=1, linestyle='--')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#58a6ff')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve', color='#58a6ff', fontsize=13)
axes[0].legend(loc='lower right', facecolor='#161b22', edgecolor='#30363d')
axes[0].grid(True, alpha=0.3)

# ── Precision-Recall Curve ──
prec, rec, _ = precision_recall_curve(y, y_proba)
ap = average_precision_score(y, y_proba)
axes[1].plot(rec, prec, color='#bc8cff', lw=2, label=f'AP = {ap:.3f}')
axes[1].fill_between(rec, prec, alpha=0.1, color='#bc8cff')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve', color='#58a6ff', fontsize=13)
axes[1].legend(facecolor='#161b22', edgecolor='#30363d')
axes[1].grid(True, alpha=0.3)

# ── Confusion Matrix ──
cm = confusion_matrix(y, y_pred)
im = axes[2].imshow(cm, cmap='Blues', aspect='auto')
axes[2].set_xticks([0,1]); axes[2].set_yticks([0,1])
axes[2].set_xticklabels(['Safe', 'Vulnerable'])
axes[2].set_yticklabels(['Safe', 'Vulnerable'])
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')
axes[2].set_title('Confusion Matrix', color='#58a6ff', fontsize=13)
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, str(cm[i,j]), ha='center', va='center',
                     color='white', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig('../datasets/model_performance.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

In [ ]:
# CV scores per fold
fig, ax = plt.subplots(figsize=(8, 4))
folds = [f'Fold {i+1}' for i in range(5)]
x = np.arange(5)
width = 0.25
ax.bar(x - width, cv_auc,  width, label='ROC-AUC',  color='#58a6ff', alpha=0.85)
ax.bar(x,         cv_f1,   width, label='F1 Score',  color='#bc8cff', alpha=0.85)
ax.bar(x + width, cv_acc,  width, label='Accuracy',  color='#2ed573', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(folds)
ax.set_ylim(0, 1.1)
ax.set_title('Cross-Validation Scores per Fold', color='#58a6ff', fontsize=13)
ax.legend(facecolor='#161b22', edgecolor='#30363d')
ax.axhline(0.8, color='#ffa502', linestyle='--', linewidth=1, alpha=0.5, label='0.8 threshold')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('../datasets/cv_scores.png', dpi=150, bbox_inches='tight', facecolor='#0d1117')
plt.show()

## 5. SHAP Explainability

In [ ]:
try:
    import shap
    shap.initjs()

    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X)

    # For binary classification, shap_values may be a list
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values

    print('SHAP values computed ✓')
    print(f'Shape: {sv.shape}')

    # Global feature importance
    mean_abs_shap = np.abs(sv).mean(axis=0)
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'mean_abs_shap': mean_abs_shap
    }).sort_values('mean_abs_shap', ascending=False)

    print('\nTop 10 most important features (SHAP):')
    print(importance_df.head(10).to_string(index=False))

except ImportError:
    print('SHAP not installed. Run: pip install shap')
    print('Falling back to built-in feature importance...')
    if hasattr(model, 'feature_importances_'):
        importance_df = pd.DataFrame({
            'feature': feature_names,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        print(importance_df.head(10).to_string(index=False))

In [ ]:
# Feature importance bar chart
top_n = importance_df.head(10)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(top_n['feature'][::-1], top_n.iloc[:, 1].values[::-1],
               color='#58a6ff', edgecolor='#30363d', linewidth=0.5)
ax.set_title('Top 10 Feature Importances (SHAP / Built-in)',
             color='#58a6ff', fontsize=13)
ax.set_xlabel('Mean |SHAP value| or Importance')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('../datasets/shap_importance.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()

In [ ]:
# Per-sample SHAP explanation for a vulnerable snippet
try:
    vuln_idx = np.where(y == 1)[0][0]
    sample_code = df.iloc[vuln_idx]['code_snippet']
    print(f'Explaining prediction for:\n  {sample_code[:80]}...')

    sample_shap = sv[vuln_idx]
    sample_feat = pd.Series(X[vuln_idx], index=feature_names)

    explanation_df = pd.DataFrame({
        'feature': feature_names,
        'value': X[vuln_idx],
        'shap': sample_shap
    }).sort_values('shap', ascending=False)

    print('\nPer-feature SHAP contributions (top 8):')
    print(explanation_df.head(8).to_string(index=False))
except Exception as e:
    print(f'SHAP per-sample explanation skipped: {e}')

## 6. Model Export

In [ ]:
import joblib, json
from pathlib import Path

models_dir = Path('../models')
models_dir.mkdir(exist_ok=True)

# Save model
model_path = models_dir / 'vuln_classifier.joblib'
joblib.dump(model, model_path)

# Save metadata
meta = {
    'version': f'{model_name.lower()}-v1.0',
    'n_samples': len(X),
    'n_features': X.shape[1],
    'cv_auc': float(cv_auc.mean()),
    'cv_auc_std': float(cv_auc.std()),
    'cv_f1': float(cv_f1.mean()),
    'train_auc': float(roc_auc_score(y, y_proba)),
    'feature_names': feature_names,
    'model_type': model_name,
}
(models_dir / 'model_meta.json').write_text(json.dumps(meta, indent=2))

print(f'Model saved: {model_path} ({model_path.stat().st_size:,} bytes)')
print(f'Metadata saved: {models_dir / "model_meta.json"}')
print(f'\nFinal metrics summary:')
print(f'  CV ROC-AUC:  {cv_auc.mean():.4f} ± {cv_auc.std():.4f}')
print(f'  CV F1:       {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'  Train AUC:   {roc_auc_score(y, y_proba):.4f}')
print(f'\nModel ready for use with: vulnscanner scan /path/to/code')

## 7. Live Prediction Demo

In [ ]:
from vulnscanner.ml.classifier import VulnClassifier

clf = VulnClassifier()

test_snippets = [
    ('cursor.execute("SELECT * FROM users WHERE id = " + user_id)', 'python', 'SQLi (vuln)'),
    ('cursor.execute("SELECT * FROM users WHERE id = %s", (user_id,))', 'python', 'Parameterised (safe)'),
    ('os.system(f"ping {host}")', 'python', 'CMDi (vuln)'),
    ('subprocess.run(["ping", "-c", "1", host], check=True)', 'python', 'Safe subprocess'),
    ('SECRET_KEY = "supersecretkey123"', 'python', 'Hardcoded secret (vuln)'),
    ('SECRET_KEY = os.environ["SECRET_KEY"]', 'python', 'Env var (safe)'),
    ('requests.get(request.args["url"])', 'python', 'SSRF (vuln)'),
    ('hashlib.md5(password.encode()).hexdigest()', 'python', 'Weak hash (vuln)'),
    ('bcrypt.hashpw(password.encode(), bcrypt.gensalt())', 'python', 'bcrypt (safe)'),
    ('jwt.decode(token, options={"verify_signature": False})', 'python', 'JWT no verify (vuln)'),
]

print(f'{"Code":<55} {"Expected":<22} {"Prediction":<10} {"Confidence"}')
print('-' * 105)
for code, lang, expected in test_snippets:
    label, conf = clf.predict(code, lang)
    pred_str = 'VULNERABLE' if label == 1 else 'SAFE'
    correct = '✓' if ('vuln' in expected.lower()) == (label == 1) else '✗'
    print(f'{correct} {code[:52]:<54} {expected:<22} {pred_str:<10} {conf:.2f}')